In [19]:
print("OK!")

OK!


In [ ]:
from src.helper import download_hugging_face_embeddings
from langchain_pinecone import PineconeVectorStore
from langchain_openai import OpenAI
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate
from langchain_community.llms import CTransformers
from langchain_classic.chains import RetrievalQA
from dotenv import load_dotenv
# from src.prompt import *
import os


: 

In [ ]:
PINECONE_API_KEY = os.environ.get("PINECONE_API_KEY")   
PINECONE_API_ENV = "gcp-starter"

In [5]:
%pwd

'c:\\Users\\dawod\\Desktop\\eCHIS_chatBot\\eCHIS-Assistant-AIChatBot\\Notebooks'

In [3]:
import os
os.chdir("../")

In [4]:
from langchain.document_loaders import PyPDFLoader, DirectoryLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

In [5]:
#Extract data from the PDF
def load_pdf(data):
    loader = DirectoryLoader(data,
                    glob="*.pdf",
                    loader_cls=PyPDFLoader)
    
    documents = loader.load()

    return documents

In [6]:
extracted_data = load_pdf("Data/")

could not convert string to float: b'0.000000000000-5684342' : FloatObject (b'0.000000000000-5684342') invalid; use 0.0 instead
could not convert string to float: b'0.000000000000-5684342' : FloatObject (b'0.000000000000-5684342') invalid; use 0.0 instead


In [ ]:
# extracted_data

[Document(page_content=' \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \neCHIS Implementation \nManual  \n \n \n \nApril, 2021 \n Policy plan Monitoring and Directorate \nAddis Ababa, Ethiopia \n ', metadata={'source': 'Data\\knowledge_base.pdf', 'page': 0}),
 Document(page_content=' \n \nContents \nFOREWORD ........................................................................................................................................................... 2 \nAcknowledgement ................................................................................................................................................ 3 \nAcronyms: ............................................................................................................................................................. 4 \n1. Background ................................................................................................................................................... 5 \n2.  Operational defini

In [7]:
#Create text chunks
def text_split(extracted_data):
    text_splitter = RecursiveCharacterTextSplitter(chunk_size = 500, chunk_overlap = 20)
    text_chunks = text_splitter.split_documents(extracted_data)

    return text_chunks

In [8]:
text_chunks = text_split(extracted_data)
print("length of my chunk:", len(text_chunks))

length of my chunk: 1781


In [9]:
 text_chunks

[Document(page_content='eCHIS Implementation \nManual  \n \n \n \nApril, 2021 \n Policy plan Monitoring and Directorate \nAddis Ababa, Ethiopia', metadata={'source': 'Data\\knowledge_base.pdf', 'page': 0}),
 Document(page_content='Contents \nFOREWORD ........................................................................................................................................................... 2 \nAcknowledgement ................................................................................................................................................ 3', metadata={'source': 'Data\\knowledge_base.pdf', 'page': 1}),
 Document(page_content='Acronyms: ............................................................................................................................................................. 4 \n1. Background ................................................................................................................................................... 5', m

In [27]:
from langchain.embeddings import HuggingFaceEmbeddings

In [24]:
#download embedding model
def download_hugging_face_embeddings():
    embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
    return embeddings

In [25]:
embeddings = download_hugging_face_embeddings()

ImportError: Could not import sentence_transformers python package. Please install it with `pip install sentence_transformers`.

In [ ]:
embeddings

In [ ]:
query_result = embeddings.embed_query("Hello world")
print("Length", len(query_result))

In [ ]:
# query_result

In [ ]:
#Initializing the Pinecone
pinecone.init(api_key=PINECONE_API_KEY,
              environment=PINECONE_API_ENV)

index_name="medical-chatbot"

#Creating Embeddings for Each of The Text Chunks & storing
docsearch=Pinecone.from_texts([t.page_content for t in text_chunks], embeddings, index_name=index_name)

In [ ]:
#If we already have an index we can load it like this
docsearch=Pinecone.from_existing_index(index_name, embeddings)

query = "What are Allergies"

docs=docsearch.similarity_search(query, k=3)

print("Result", docs)

In [ ]:
prompt_template="""
Use the following pieces of information to answer the user's question.
If you don't know the answer, just say that you don't know, don't try to make up an answer.

Context: {context}
Question: {question}

Only return the helpful answer below and nothing else.
Helpful answer:
"""

In [ ]:
PROMPT=PromptTemplate(template=prompt_template, input_variables=["context", "question"])
chain_type_kwargs={"prompt": PROMPT}

In [ ]:
llm=CTransformers(model="model/llama-2-7b-chat.ggmlv3.q4_0.bin",
                  model_type="llama",
                  config={'max_new_tokens':512,
                          'temperature':0.8})

In [ ]:
qa=RetrievalQA.from_chain_type(
    llm=llm, 
    chain_type="stuff", 
    retriever=docsearch.as_retriever(search_kwargs={'k': 2}),
    return_source_documents=True, 
    chain_type_kwargs=chain_type_kwargs)

In [ ]:
while True:
    user_input=input(f"Input Prompt:")
    result=qa({"query": user_input})
    print("Response : ", result["result"])